# Qwen3 Layer Export with Quantization

## Pipeline:
1. Download full model (~8GB)
2. Add layer hooks, test layer capture
3. Quantize to INT4 (~2.4GB)
4. Save both versions to Google Drive

**Why two versions?**
- Full model: for layer modification (hooks work with full precision)
- Quantized model: for efficient inference in EVA

In [ ]:
# Mount Google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = "/content/drive/MyDrive/EVA_Ai_Exports"
FULL_MODEL_DIR = "/content/full_model"
QUANTIZED_DIR = os.path.join(OUTPUT_DIR, "qwen_layer_model_4bit")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FULL_MODEL_DIR, exist_ok=True)
os.makedirs(QUANTIZED_DIR, exist_ok=True)

print(f"Output: {OUTPUT_DIR}")

In [ ]:
# Install dependencies
!pip install -q transformers torch huggingface_hub hf_transfer bitsandbytes accelerate

In [ ]:
# Enable fast downloads
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import snapshot_download
import torch

MODEL_ID = "RefalMachine/RuadaptQwen3-4B-Instruct"
NUM_LAYERS = 36
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")

In [ ]:
# STEP 1: Download full model
print("=" * 60)
print("STEP 1: Downloading full model...")
print("=" * 60)

print(f"Downloading {MODEL_ID}...")
local_path = snapshot_download(
    repo_id=MODEL_ID,
    local_dir=FULL_MODEL_DIR,
    local_dir_use_symlinks=False,
    resume_download=True
)
print(f"Downloaded to: {local_path}")

In [ ]:
# STEP 2: Load full model and add layer hooks
print("=" * 60)
print("STEP 2: Loading full model and adding layer hooks...")
print("=" * 60)

from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer

config = AutoConfig.from_pretrained(FULL_MODEL_DIR, trust_remote_code=True)
print(f"hidden_size: {config.hidden_size}")
print(f"num_layers: {config.num_hidden_layers}")

tokenizer = AutoTokenizer.from_pretrained(FULL_MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading full model (bfloat16)...")
full_model = AutoModelForCausalLM.from_pretrained(
    FULL_MODEL_DIR,
    config=config,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
    low_cpu_mem_usage=True
)
full_model.eval()
print("Full model loaded!")

In [ ]:
# Create layer capture wrapper for FULL model
class ModelWithLayerOutputs(torch.nn.Module):
    def __init__(self, base_model, num_layers):
        super().__init__()
        self.base_model = base_model
        self.num_layers = num_layers
        self.layer_outputs = {}
        
        for idx, layer in enumerate(base_model.model.layers):
            layer.register_forward_hook(self._hook_fn(idx))

    def _hook_fn(self, layer_idx):
        def hook(module, input, output):
            self.layer_outputs[layer_idx] = output[0].detach()
        return hook

    def forward(self, input_ids, attention_mask=None):
        self.layer_outputs.clear()
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        
        all_outputs = {}
        all_outputs[-1] = self.base_model.model.embed_tokens(input_ids).detach()
        for idx in range(self.num_layers):
            if idx in self.layer_outputs:
                all_outputs[idx] = self.layer_outputs[idx]
        all_outputs['final'] = outputs.logits
        return all_outputs

In [ ]:
# Test layer capture on FULL model
print("Testing layer capture on full model...")

wrapped_model = ModelWithLayerOutputs(full_model, NUM_LAYERS)
wrapped_model.eval()

test_input = tokenizer("Привет", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    outputs = wrapped_model(test_input['input_ids'], test_input['attention_mask'])

print(f"Embedding output shape: {outputs[-1].shape}")
for layer_idx in sorted([k for k in outputs.keys() if isinstance(k, int)])[:5]:
    print(f"Layer {layer_idx} output shape: {outputs[layer_idx].shape}")
print(f"Final logits shape: {outputs['final'].shape}")

print("Layer capture works!")

In [ ]:
# Save full model checkpoint with layer info
print("Saving full model checkpoint...")

pt_path = os.path.join(OUTPUT_DIR, 'qwen_layer_model_full.pt')

torch.save({
    'model_state_dict': wrapped_model.state_dict(),
    'config': config,
    'num_layers': NUM_LAYERS
}, pt_path)

print(f"Saved: {pt_path}")
!ls -lh $pt_path

In [ ]:
# STEP 3: Quantize the model to INT4
print("=" * 60)
print("STEP 3: Quantizing to INT4...")
print("=" * 60)

# Clear memory first
import gc
del full_model, wrapped_model
gc.collect()
torch.cuda.empty_cache()

from transformers import BitsAndBytesConfig

# INT4 Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading model with INT4 quantization...")
quantized_model = AutoModelForCausalLM.from_pretrained(
    FULL_MODEL_DIR,
    config=config,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
quantized_model.eval()
print("Quantized model loaded!")

if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
# Test quantized model inference
print("Testing quantized model...")

test_input = tokenizer("Привет", return_tensors="pt")
test_input = {k: v.to(quantized_model.device) for k, v in test_input.items()}

with torch.no_grad():
    outputs = quantized_model(**test_input)

print(f"Logits shape: {outputs.logits.shape}")

# Note: quantized model uses output_hidden_states differently
with torch.no_grad():
    outputs = quantized_model(**test_input, output_hidden_states=True)
    hidden_states = outputs.hidden_states
    print(f"Hidden states count: {len(hidden_states)}")
    print(f"Layer 0 shape: {hidden_states[0].shape}")

In [ ]:
# STEP 4: Save quantized model to Google Drive
print("=" * 60)
print("STEP 4: Saving quantized model to Google Drive...")
print("=" * 60)

print(f"Saving to: {QUANTIZED_DIR}")

# Save quantized model (creates model.safetensors + config)
quantized_model.save_pretrained(QUANTIZED_DIR)
tokenizer.save_pretrained(QUANTIZED_DIR)

print("Model saved!")
!ls -lh $QUANTIZED_DIR
!du -sh $QUANTIZED_DIR

In [ ]:
# Save layer info for EVA
print("Saving layer capture info...")

layer_info = {
    'num_layers': NUM_LAYERS,
    'hidden_size': config.hidden_size,
    'is_quantized': True,
    'quantization_type': 'int4',
    'quantized_model_path': QUANTIZED_DIR,
    'full_model_path': OUTPUT_DIR + '/qwen_layer_model_full.pt'
}

torch.save(layer_info, os.path.join(QUANTIZED_DIR, 'layer_info.pt'))

print("Layer info saved!")

In [ ]:
# Final summary
print("=" * 60)
print("COMPLETE!")
print("=" * 60)
print("\nSaved files:")
print("1. Full model (8GB):")
!ls -lh $OUTPUT_DIR/qwen_layer_model_full.pt
print("\n2. Quantized model (2.4GB):")
!ls -lh $QUANTIZED_DIR
print("\n3. Layer info:")
!ls -lh $QUANTIZED_DIR/layer_info.pt
print("\n" + "=" * 60)